# 04 — Parallel & Multi-Tool Calls — Practical Examples

**Covers**: parallel execution with asyncio, fan-out pattern, sequential vs parallel latency comparison

In [ ]:
import os, json, asyncio, time
from openai import OpenAI, AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
async_client = AsyncOpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Ready')

## 1. Trigger Parallel Tool Calls from LLM

In [ ]:
TOOLS = [
  {'type':'function','function':{'name':'get_stock','description':'Get stock price for a ticker.','parameters':{'type':'object','properties':{'ticker':{'type':'string'}},'required':['ticker']}}},
  {'type':'function','function':{'name':'get_news','description':'Get latest news for a company.','parameters':{'type':'object','properties':{'company':{'type':'string'}},'required':['company']}}},
]

def get_stock(ticker: str) -> str:
    return {'AAPL': '$185.50', 'TSLA': '$248.00', 'MSFT': '$420.00'}.get(ticker, '$100.00')

def get_news(company: str) -> str:
    return f'{company}: [Mock] New product launch announced, strong Q4 guidance.'

TOOL_MAP = {'get_stock': get_stock, 'get_news': get_news}

r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Get stock price AND latest news for Apple and Tesla simultaneously'}],
    tools=TOOLS,
    parallel_tool_calls=True
)
msg = r.choices[0].message
print(f'Number of parallel tool calls: {len(msg.tool_calls)}')
for tc in msg.tool_calls:
    print(f'  - {tc.function.name}({tc.function.arguments})')

## 2. Execute All Parallel Calls Concurrently

In [ ]:
async def async_tool(tc) -> dict:
    await asyncio.sleep(0.1)  # Simulate async I/O
    args = json.loads(tc.function.arguments)
    result = TOOL_MAP[tc.function.name](**args)
    return {'role': 'tool', 'tool_call_id': tc.id, 'content': result}

async def run_parallel_agent(user_input: str) -> str:
    messages = [{'role': 'user', 'content': user_input}]
    for step in range(5):
        r = await async_client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, parallel_tool_calls=True
        )
        msg = r.choices[0].message
        messages.append(msg)
        if r.choices[0].finish_reason == 'tool_calls':
            t0 = time.time()
            results = await asyncio.gather(*[async_tool(tc) for tc in msg.tool_calls])
            elapsed = time.time() - t0
            print(f'Step {step+1}: Ran {len(results)} tools in {elapsed:.2f}s (parallel)')
            messages.extend(results)
        else:
            return msg.content
    return 'done'

result = await run_parallel_agent('Get stock AND news for Apple at the same time')
print(f'\nAnswer: {result}')

## 3. Sequential vs Parallel Latency Comparison

In [ ]:
import concurrent.futures

tickers = ['AAPL', 'TSLA', 'MSFT', 'GOOGL', 'NVDA']

def simulate_api_call(ticker: str) -> str:
    time.sleep(0.3)  # Simulate 300ms API latency
    return f'{ticker}: $100'

# Sequential
t0 = time.time()
for t in tickers:
    simulate_api_call(t)
seq_time = time.time() - t0

# Parallel with ThreadPoolExecutor
t0 = time.time()
with concurrent.futures.ThreadPoolExecutor() as ex:
    list(ex.map(simulate_api_call, tickers))
par_time = time.time() - t0

print(f'Tools: {len(tickers)} | Simulated latency: 300ms each')
print(f'Sequential: {seq_time:.2f}s')
print(f'Parallel:   {par_time:.2f}s')
print(f'Speedup:    {seq_time/par_time:.1f}x faster')